In [90]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [91]:
df = pd.read_csv('moviereviews.csv')

In [92]:
df.head()

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
2,pos,this has been an extraordinary year for austra...
3,pos,according to hollywood movies made in last few...
4,neg,my first press screening of 1998 and already i...


In [93]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   2000 non-null   object
 1   review  1965 non-null   object
dtypes: object(2)
memory usage: 31.4+ KB


In [94]:
df['label'].value_counts()

label
neg    1000
pos    1000
Name: count, dtype: int64

In [95]:
print(df['review'][1])

some talented actresses are blessed with a demonstrated wide acting range while others , almost as gifted , have more limited types of parts for which they are suitable . 
as was amply evident after basic instinct , sharon stone can play sensual roles with great abandon . 
rejecting her natural abilities , she has spent the rest of her entire career trying with little success to play against type . 
gloria is her latest disaster . 
babe ruth didn't quit baseball after one season to play football in a quixotic quest to prove his athletic dexterity , and neither should stone reject what she does best . 
janeane garofalo , for example , is no less wonderful an actress because she could have never pulled off stone's part in basic instinct ; neither is stone any less talented because she couldn't do garofalo's comedic roles . 
gloria , directed by respected director sidney lumet and adapted by steve antin from the 1980 screenplay by john cassavetes , was not screened in advance for critics 

In [96]:
df.isnull().sum()

label      0
review    35
dtype: int64

In [97]:
df = df.dropna()

In [98]:
df['review'].value_counts()

review
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [99]:
df[df['review'] == '  '] 

,label,review
57,neg,
71,pos,
147,pos,
151,pos,
283,pos,
307,pos,
313,neg,
323,pos,
343,pos,
351,neg,


In [100]:
df['review'].str.isspace()

0       False
1       False
2       False
3       False
4       False
        ...  
1995    False
1996    False
1997    False
1998    False
1999    False
Name: review, Length: 1965, dtype: bool

In [101]:
df[df['review'].str.isspace()]

,label,review
57,neg,
71,pos,
147,pos,
151,pos,
283,pos,
307,pos,
313,neg,
323,pos,
343,pos,
351,neg,


In [102]:
df = df[~df['review'].str.isspace()]

In [103]:
df['label'].value_counts()

label
neg    969
pos    969
Name: count, dtype: int64

### Exploration on bag of words: 
* trying to see the common words in positive and negative reviews. 

In [104]:
from sklearn.feature_extraction.text import CountVectorizer


In [105]:
cv = CountVectorizer(stop_words='english')

In [106]:
matrix = cv.fit_transform(df[df['label'] == 'neg']['review'])
freq = list(zip(cv.get_feature_names_out(),matrix.sum(axis = 0).tolist()[0]))
print("Top 20 repetative words in Negative reviews.")
sorted(freq, key = lambda x:-x[1])[:20]

Top 20 repetative words in Negative reviews.


[('film', 4063),
 ('movie', 3131),
 ('like', 1808),
 ('just', 1480),
 ('time', 1127),
 ('good', 1117),
 ('bad', 997),
 ('character', 926),
 ('story', 908),
 ('plot', 888),
 ('characters', 838),
 ('make', 813),
 ('really', 743),
 ('way', 734),
 ('little', 696),
 ('don', 683),
 ('does', 666),
 ('doesn', 648),
 ('action', 635),
 ('scene', 634)]

In [107]:
matrix = cv.fit_transform(df[df['label'] == 'pos']['review'])
freq = list(zip(cv.get_feature_names_out(),matrix.sum(axis = 0).tolist()[0]))
print("Top 20 repetative words in Postive reviews.")
sorted(freq, key = lambda x:-x[1])[:20]

Top 20 repetative words in Postive reviews.


[('film', 5002),
 ('movie', 2389),
 ('like', 1721),
 ('just', 1273),
 ('story', 1199),
 ('good', 1193),
 ('time', 1175),
 ('character', 1037),
 ('life', 1032),
 ('characters', 957),
 ('way', 864),
 ('films', 851),
 ('does', 828),
 ('best', 788),
 ('people', 769),
 ('make', 764),
 ('little', 751),
 ('really', 731),
 ('man', 728),
 ('new', 702)]

### Training and Prediction:

In [108]:
from sklearn.model_selection import train_test_split

X = df['review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=50)

In [109]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

In [110]:
pipe = Pipeline([('tfidf', TfidfVectorizer()),('svc',LinearSVC())])

In [111]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('svc', LinearSVC())])

In [112]:
from sklearn.metrics import classification_report, confusion_matrix

In [113]:
pred = pipe.predict(X_test)

In [114]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

         neg       0.89      0.80      0.84       205
         pos       0.79      0.89      0.84       183

    accuracy                           0.84       388
   macro avg       0.84      0.84      0.84       388
weighted avg       0.84      0.84      0.84       388



In [115]:
confusion_matrix(pred, y_test)

array([[163,  21],
       [ 42, 162]])

### Final training with entire data available:

In [116]:
pipe.fit(df['review'], df['label'])

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('svc', LinearSVC())])

In [119]:
new_review = ['Nice movie. Great acting and writing.']
pipe.predict(new_review)

array(['pos'], dtype=object)

In [120]:
new_review = ['Nice movie. Great acting.']
pipe.predict(new_review)

array(['neg'], dtype=object)

In [121]:
new_review = ["Pawan Kalyan's latest film, OG, is a refreshing and captivating experience that stands out from the usual fare. The film’s greatest strength lies in its bold and experimental approach to filmmaking, a welcome change that shows a dedication to pushing cinematic boundaries. The director's vision is evident in every frame, creating a stylish and immersive world that pulls the audience in from the very first scene."]
pipe.predict(new_review)

array(['pos'], dtype=object)

In [122]:
new_review = ["The worst and the very worst film in the universe if you ask me. Zero zip nada cocept and the worst acting I've ever seen. Congrats Mr.Vijay I hope you encounter more ironies like you are a good actor"]
pipe.predict(new_review)

array(['neg'], dtype=object)